In [1]:
import os
import glob

import numpy as np

from scipy.interpolate import interp1d

import bagpipes as pipes

import astropy.units as u
from astropy.io import fits
from astropy.table import Table, vstack

from astropy.coordinates import SkyCoord

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

import h5py

import sys

sys.path.append(os.path.abspath('..'))

from mrileyowens.dja import cutout, check
from mrileyowens.bagpipes import plot

In [ ]:
home = os.getcwd()
data = f'{home}/data'
figs = f'{home}/figs'
results = f'{home}/results'

def drop_f070w():

    '''
    Apply the initial F070W dropout (z ~ 6) selections to the catalogs
    '''

    catalogs = glob.glob(f'{data}/catalogs/photometryCatalog_*_*2026.fits')

    for catalog in catalogs:

        print(os.path.basename(catalog))

        # Open the field catalog
        hdul = fits.open(catalog)
        print(hdul[1].columns)

        # Open the merged catalog's photometry as a table
        t = Table(hdul[1].data)
    
        # ---------------------------------------------------------------------------------------------------------------------------------------
        # Set the photometry of the Lyman-break shortward filters used to calculate a dropout color to the 1-sigma upper limit when their SNR < 1
        # ---------------------------------------------------------------------------------------------------------------------------------------

        # Directly set the F070W flux density as the uncertainty when the SNR is < 1
        t['f070w_tot_0'] = np.where(t['f070w_tot_0'] / t['f070w_etot_0'] < 1, t['f070w_etot_0'], t['f070w_tot_0'])

        # Before adjusting the F606W photometry, calculate the F606W SNR. This is necessary to do before the next operation assigning upper limits 
        # to low SNR F606W photometry, which would alter the F606W SNR if measured after that operation.
        f606w_snr = t['f606w_tot_0'] / t['f606w_etot_0']

        # Directly set the F606W flux density as the uncertainty when the SNR is < 1
        t['f606w_tot_0'] = np.where(t['f606w_tot_0'] / t['f606w_etot_0'] < 1, t['f606w_etot_0'], t['f606w_tot_0'])

        # ----------------------------------------------------------
        # Create boolean masks of the initial photometric selections
        # ----------------------------------------------------------

        #print(t['f606w_tot_0'])
        #print(t['f090w_tot_0'])

        # Calculate AB magnitudes for photometry used to measure colors. Use the unitless values since including units has been troublesome.
        f606w_ABmag = (t['f606w_tot_0'] * u.nJy).to(u.ABmag).value
        f070w_ABmag = (t['f070w_tot_0'] * u.nJy).to(u.ABmag).value
        f090w_ABmag = (t['f090w_tot_0'] * u.nJy).to(u.ABmag).value
        f150w_ABmag = (t['f150w_tot_0'] * u.nJy).to(u.ABmag).value

        # Make a boolean mask for a break in F775W
        cond_f070w_break = (f070w_ABmag - f090w_ABmag) > 1.2

        # Make a boolean mask for a much flatter near-IR 
        cond_flat = (f090w_ABmag - f150w_ABmag) < 1.0

        # Make a boolean mask for a much sharper F775W break than the observed near-IR slope
        cond_f070w_break_gtr = (f070w_ABmag - f090w_ABmag) > (f090w_ABmag - f150w_ABmag + 1.2)

        # Make a boolean mask for a low SNR in F435W. At z ~ 6, the Lyman break is completely longward of F435W, so this filter should have minimal 
        # flux for true Lyman break galaxies.
        cond_lyc_snr = (t['f435w_tot_0'] / t['f435w_etot_0']) < 2

        # Make a boolean mask for a strong break in F606W, which z ~ 6 galaxies should show. If the SNR in F606W is low (< 2), relax the necessary 
        # break condition.
        cond_f606w_break = (f606w_ABmag - f090w_ABmag) > 2.7
        cond_f606w_break_weak = (f606w_ABmag - f090w_ABmag) > 1.8
        cond_f606w_break = np.where(f606w_snr < 2, cond_f606w_break_weak, cond_f606w_break)

        # Make a boolean mask for strong F775W breaks
        cond_f070w_break_strong = (f070w_ABmag - f090w_ABmag) > 2.5

        # ------------------------------------------------------
        # Create boolean masks of the photometric SNR selections
        # ------------------------------------------------------

        # Make a list of JWST filters in the joint JADES GOODS-N / GOODS-S catalog
        jwst_filters = ['f070w','f090w','f115w','f150w','f200w','f277w','f335m','f356w','f410m','f444w']

        # Make empty lists of the JWST photometry and uncertainties corresponding to the filters, to be filled in the loop below
        jwst_photometry = []
        jwst_photometry_e = []

        # For each JWST filter in the catalog
        for filter in jwst_filters:

            # Add the filter's photometry to the lists
            jwst_photometry.append([t[f'{filter}_tot_0'].data])
            jwst_photometry_e.append([t[f'{filter}_etot_0'].data])

        # Stack the JWST photometry and uncertainties
        jwst_photometry = np.vstack(jwst_photometry)
        jwst_photometry_e = np.vstack(jwst_photometry_e)

        # Calculate the SNR of the JWST photometry
        jwst_photometry_snr = jwst_photometry / jwst_photometry_e

        # Make a boolean mask for objects with any JWST filter with SNR > 5
        cond_snr_gtr_5 = np.any(jwst_photometry_snr > 5, axis=0)

        # Make a boolean mask for objects with at least 3 JWST filters with SNR > 3
        cond_3_snr_gtr_3 = np.sum(jwst_photometry_snr > 3, axis=0) >= 3

        # Make boolean masks for objects with F814W / F850LP SNRs > 3
        cond_f814w_snr_gtr_3 = (t['f814w_tot_0'] / t['f814w_etot_0']) > 3
        #cond_f850lp_snr_gtr_3 = (t['f850LP_KRON_S'] / t['F850LP_KRON_S_e']) > 3

        #cond_real_errors = ~np.isnan(t['F435W_KRON_S_e']) & ~np.isnan(t['F775W_KRON_S_e']) & (~np.isnan(t['F814W_KRON_S_e']) | ~np.isnan(t['F850LP_KRON_S_e'])) & ~np.isnan(t['F090W_KRON_S_e']) & ~np.isnan(t['F150W_KRON_S_e'])
        cond_real_errors = ~np.isnan(t['f070w_tot_0']) & ~np.isnan(t['f814w_etot_0']) & ~np.isnan(t['f090w_etot_0']) & ~np.isnan(t['f150w_etot_0'])

        # ---------------------------------------------------
        # Create boolean masks for artifacts or contamination
        # ---------------------------------------------------

        #cond_real_errors = ~np.isnan(t['F775W_KRON_S_e']) & (~np.isnan(t['F814W_KRON_S_e']) | ~np.isnan(t['F850LP_KRON_S_e'])) & ~np.isnan(t['F090W_KRON_S_e']) & ~np.isnan(t['F150W_KRON_S_e'])

        # Make a boolean mask of non-stars
        #cond_star = Table(hdul['FLAG'].data)['FLAG_ST'] != 1

        # Make a boolean mask of objects not contaminated by bright stars
        #cond_bs = Table(hdul['FLAG'].data)['FLAG_BS'] != 1.0

        # Make a boolean mask of objects that don't have a bright neighbor >10x as bright
        #cond_bn = Table(hdul['FLAG'].data)['FLAG_BN'] != 2.0

        # ----------------------------------------------
        # Combine and apply the individual boolean masks
        # ----------------------------------------------

        # Combine all the boolean masks into a single mask
        conditions = (cond_real_errors & cond_f070w_break & cond_flat & cond_f070w_break_gtr & ((cond_lyc_snr & cond_f606w_break) | cond_f070w_break_strong) & cond_snr_gtr_5 & cond_3_snr_gtr_3 & cond_f814w_snr_gtr_3 )

        print(t['f606w_tot_0'][conditions])
        print(t['f090w_tot_0'][conditions])

        # Mask the table
        t = t[conditions]
        print(len(t))

        # Make a HDU from the table
        tab = fits.BinTableHDU(t.as_array())

        # Make a HDUL from the table and save it
        hdul = fits.HDUList([fits.PrimaryHDU(), tab])
        hdul.writeto(f'{results}/catalogs/{os.path.basename(catalog).split('_')[1]}_f070w_dropouts_init.fits', overwrite=True)

def drop_f090w():

    '''
    Apply the F090W dropout (z ~ 6) selection to the catalogs
    '''

    # Get the file paths to the photometric catalogs of the fields
    catalogs = glob.glob(f'{data}/catalogs/photometryCatalog_*_*2026.fits')

    # For each catalog
    for catalog in catalogs:

        # Open the field catalog
        hdul = fits.open(catalog)

        # Open the merged catalog's photometry as a table
        t = Table(hdul[1].data)

        # Get the filters in the catalog
        filters = [col.split('_')[0] for col in hdul[1].columns.names if col.endswith('_tot_0')]

        filters_required = ['f090w','f115w','f150w','f200w','f277w','f356w','f410m','f444w']

        # Get the catalog's field as a string
        field = os.path.basename(catalog).split('_')[1]

        # -----------------------------------------
        # Create a boolean mask for compact objects
        # -----------------------------------------

        # Make an all-true mask matching the number of sources in the catalog
        cond_compact = np.ones(np.shape(t['ID']), dtype=bool)

        # For each of several filters
        for filter in ['f115w','f150w','f200w']:

            # Require that the sources are compact in that filter
            cond_compact &= (t[f'{filter.upper()}_peakPixelSNR'] > 2.5)

        # For each aperture size
        for aper in range(3):

            # ---------------------------------------------------------------------------------------------------------------------------------------
            # Set the photometry of the Lyman-break shortward filters used to calculate a dropout color to the 1-sigma upper limit when their SNR < 1
            # ---------------------------------------------------------------------------------------------------------------------------------------

            # Directly set the F070W flux density as the uncertainty when the SNR is < 1
            t[f'f090w_tot_{aper}'] = np.where(t[f'f090w_tot_{aper}'] / t[f'f090w_etot_{aper}'] < 1, t[f'f090w_etot_{aper}'], t[f'f090w_tot_{aper}'])

            # ----------------------------------------------------------
            # Create boolean masks of the initial photometric selections
            # ----------------------------------------------------------

            # Calculate AB magnitudes for photometry used to measure colors. Use the unitless values since including units has been troublesome.
            f090w_ABmag = (t[f'f090w_tot_{aper}'] * u.nJy).to(u.ABmag).value
            f115w_ABmag = (t[f'f115w_tot_{aper}'] * u.nJy).to(u.ABmag).value
            f200w_ABmag = (t[f'f200w_tot_{aper}'] * u.nJy).to(u.ABmag).value

            # Make a boolean mask for a break in F070W
            cond_f090w_break = (f090w_ABmag - f115w_ABmag) > 1.5

            # Make a boolean mask for a much flatter near-IR 
            cond_flat = (f115w_ABmag - f200w_ABmag) < 1.0

            # Make a boolean mask for a much sharper F775W break than the observed near-IR slope
            cond_f090w_break_gtr = (f090w_ABmag - f115w_ABmag) > (f115w_ABmag - f200w_ABmag + 1.5)

            # ------------------------------------------------------
            # Create boolean masks of the photometric SNR selections
            # ------------------------------------------------------

            # Make zero-filled arrays matching the number of sources in the catalog, to be iteratively added to in the below loop
            filters_snr_gtr_3 = np.zeros(np.shape(t['ID']))
            filters_snr_gtr_5 = np.zeros(np.shape(t['ID']))

            # For each filter longward of the Lyman break at the target redshift
            for filter in [filter for filter in filters if filter not in ['f070w','f435w','f606w','f814w']]:

                # Calculate the SNR in the filter
                snr = t[f'{filter}_tot_{aper}'] / t[f'{filter}_etot_{aper}']

                # Add 1 to the number of filters with SNR > 5 or SNR > 3 if appropriate
                filters_snr_gtr_5 = np.where(snr > 5, filters_snr_gtr_5 + 1, filters_snr_gtr_5)
                filters_snr_gtr_3 = np.where(snr > 3, filters_snr_gtr_3 + 1, filters_snr_gtr_3)

            # Make a boolean mask for at least one post-Lyman break filter with SNR > 5
            cond_snr_gtr_5 = filters_snr_gtr_5 >= 1

            # Make a boolean mask for at least 3 post-Lyman break filters with SNR > 3
            cond_3_snr_gtr_3 = filters_snr_gtr_3 >= 3

            # ---------------------------------------------------------------------
            # Create a boolean mask requiring finite photometry of certain filters
            # ---------------------------------------------------------------------

            # Make a boolean mask for finite photometry
            cond_finite = np.ones(np.shape(t['ID']), dtype=bool)

            # For each of the filters requiring finite photometry
            for filter in filters_required:

                # Require the photometry in that filter be finite
                cond_finite &= ~np.isnan(t[f'{filter}_tot_{aper}'])

            # ----------------------------------------------
            # Combine and apply the individual boolean masks
            # ----------------------------------------------

            # Combine all the boolean masks into a single mask
            conditions = (cond_finite & cond_compact & cond_f090w_break & cond_flat & cond_f090w_break_gtr & cond_snr_gtr_5 & cond_3_snr_gtr_3)

            # Mask the table
            t_dropouts = t[conditions]

            # Make a HDU from the table
            tab = fits.BinTableHDU(t_dropouts.as_array())

            # Make a HDUL from the table and save it
            hdul = fits.HDUList([fits.PrimaryHDU(), tab])
            hdul.writeto(f'{results}/catalogs/{field}_f090w_dropouts_aper_{aper}.fits', overwrite=True)
            print(field, aper, len(t_dropouts))

def drop_f182m():

    '''
    Apply the F182M dropout selection to the catalogs
    '''

    # Get the file paths to the photometric catalogs of the fields
    catalogs = glob.glob(f'{data}/catalogs/photometryCatalog_*_*2026.fits')

    # For each catalog
    for catalog in catalogs:

        # Open the field catalog
        hdul = fits.open(catalog)

        # Open the merged catalog's photometry as a table
        t = Table(hdul[1].data)

        # Get the filters in the catalog
        filters = [col.split('_')[0] for col in hdul[1].columns.names if col.endswith('_tot_0')]

        filters_required = ['f090w','f115w','f150w','f182m','f200w','f210m','f277w','f356w','f410m','f444w']

        # Get the catalog's field as a string
        field = os.path.basename(catalog).split('_')[1]

        # -----------------------------------------
        # Create a boolean mask for compact objects
        # -----------------------------------------

        # Make an all-true mask matching the number of sources in the catalog
        cond_compact = np.ones(np.shape(t['ID']), dtype=bool)

        # For each of several filters
        for filter in ['f210m','f277w']:

            # Require that the sources are compact in that filter
            cond_compact &= (t[f'{filter.upper()}_peakPixelSNR'] > 3)

        # For each aperture size
        for aper in range(3):

            # ---------------------------------------------------------------------------------------------------------------------------------------
            # Set the photometry of the Lyman-break shortward filters used to calculate a dropout color to the 1-sigma upper limit when their SNR < 1
            # ---------------------------------------------------------------------------------------------------------------------------------------

            for filter in ['f090w', 'f115w', 'f150w', 'f182m']:

                # Directly set the F070W flux density as the uncertainty when the SNR is < 1
                t[f'{filter}_tot_{aper}'] = np.where(t[f'{filter}_tot_{aper}'] / t[f'{filter}_etot_{aper}'] < 1, t[f'{filter}_etot_{aper}'], t[f'{filter}_tot_{aper}'])

            # ----------------------------------------------------------
            # Create boolean masks of the initial photometric selections
            # ----------------------------------------------------------

            # Calculate AB magnitudes for photometry used to measure colors. Use the unitless values since including units has been troublesome.
            f150w_ABmag = (t[f'f150w_tot_{aper}'] * u.nJy).to(u.ABmag).value
            f182m_ABmag = (t[f'f182m_tot_{aper}'] * u.nJy).to(u.ABmag).value
            f210m_ABmag = (t[f'f210m_tot_{aper}'] * u.nJy).to(u.ABmag).value
            f277w_ABmag = (t[f'f277w_tot_{aper}'] * u.nJy).to(u.ABmag).value

            # Make a boolean mask for a break in F070W
            cond_break_1 = (f182m_ABmag - f277w_ABmag) > 1.5
            cond_break_2 = (f182m_ABmag - f210m_ABmag) > 1
            cond_break_3 = (f150w_ABmag - f210m_ABmag) > 1.5

            # Make a boolean mask for a much flatter near-IR 
            #cond_flat = (f115w_ABmag - f200w_ABmag) < 1.0

            # Make a boolean mask for a much sharper F775W break than the observed near-IR slope
            cond_break_gtr = (f182m_ABmag - f210m_ABmag) > (f210m_ABmag - f277w_ABmag + 1)

            cond_slope = (f210m_ABmag - f277w_ABmag) > -0.5

            # ------------------------------------------------------
            # Create boolean masks of the photometric SNR selections
            # ------------------------------------------------------

            # Make zero-filled arrays matching the number of sources in the catalog, to be iteratively added to in the below loop
            filters_snr_gtr_3 = np.zeros(np.shape(t['ID']))
            filters_snr_gtr_5 = np.zeros(np.shape(t['ID']))

            # For each filter longward of the Lyman break at the target redshift
            for filter in [filter for filter in filters if filter not in ['f070w','f090w','f115w','f150w','f140m','f162m','f435w','f606w','f814w']]:

                # Calculate the SNR in the filter
                snr = t[f'{filter}_tot_{aper}'] / t[f'{filter}_etot_{aper}']

                # Add 1 to the number of filters with SNR > 5 or SNR > 3 if appropriate
                filters_snr_gtr_5 = np.where(snr > 5, filters_snr_gtr_5 + 1, filters_snr_gtr_5)
                filters_snr_gtr_3 = np.where(snr > 3, filters_snr_gtr_3 + 1, filters_snr_gtr_3)

            # Make a boolean mask for at least one post-Lyman break filter with SNR > 5
            cond_snr_gtr_5 = filters_snr_gtr_5 >= 1

            # Make a boolean mask for at least 3 post-Lyman break filters with SNR > 3
            cond_3_snr_gtr_3 = filters_snr_gtr_3 >= 3

            cond_snr_low = np.ones(np.shape(t['ID']), dtype=bool)

            for filter in ['f090w','f115w','f150w']:

                cond_snr_low &= ((t[f'{filter}_tot_{aper}'] / t[f'{filter}_etot_{aper}']) < 2.5)

            # ---------------------------------------------------------------------
            # Create a boolean mask requiring finite photometry of certain filters
            # ---------------------------------------------------------------------

            # Make a boolean mask for finite photometry
            cond_finite = np.ones(np.shape(t['ID']), dtype=bool)

            # For each of the filters requiring finite photometry
            for filter in filters_required:

                # Require the photometry in that filter be finite
                cond_finite &= (~np.isnan(t[f'{filter}_tot_{aper}']) & ~np.isnan(t[f'{filter}_etot_{aper}']))

            # ----------------------------------------------
            # Combine and apply the individual boolean masks
            # ----------------------------------------------

            # Combine all the boolean masks into a single mask
            conditions = (cond_slope & cond_break_1 & cond_break_2 & cond_break_3 & cond_snr_low & cond_finite & cond_compact & cond_break_gtr & cond_snr_gtr_5 & cond_3_snr_gtr_3)

            # Mask the table
            t_dropouts = t[conditions]

            # Make a HDU from the table
            tab = fits.BinTableHDU(t_dropouts.as_array())

            # Make a HDUL from the table and save it
            hdul = fits.HDUList([fits.PrimaryHDU(), tab])
            hdul.writeto(f'{results}/catalogs/{field}_f090w_dropouts_aper_{aper}.fits', overwrite=True)
            print(field, aper, len(t_dropouts))

In [ ]:
drop_f090w()

In [ ]:
drop_f182m()